# 04 — ETA modeling with graph-enhanced ML

**Delivery ETA Intelligence** · Predictive operations & supply chain ML

---

### Business context

Last-mile and linehaul operators commit to **delivery ETAs** that drive customer NPS, SLA penalties, and dock labor planning. Map routers (OSRM) provide a baseline, but **route congestion**, **hub dwell**, and **network position** (bottleneck DCs) create systematic error.

### Why this model matters

| Value lever | Impact |
|-------------|--------|
| **Customer ETA** | Fewer missed windows → lower WISMO contacts |
| **Capacity** | Anticipate delay on high-betweenness hubs before cutoff breach |
| **Cost** | Reduce emergency linehaul and overtime on volatile corridors |
| **Risk** | SLA monitoring on lanes where prediction error is structurally high |

### Network-aware ETA prediction

Notebook **03** quantified **bottleneck hubs** and **critical lanes**. Here we inject those signals (PageRank, degree, bottleneck score, community) into supervised learning so the model knows *which nodes and corridors* tend to overrun OSRM—not only *how far* the shipment travels.

**Target:** `actual_time` (minutes) · **Grain:** segment rows with **trip-level** holdout to prevent leakage.

**Upstream:** `01` → `02` → `03` · **Artifacts:** `data/processed/delivery_logistics_processed.parquet`, `outputs/tables/03_*.csv`

## Executive framing — four questions this notebook answers

1. **Can we predict delivery ETA accurately?** — Benchmark vs linear and tree ensembles on held-out trips.
2. **Do graph features improve quality?** — Paired XGBoost ablation (tabular-only vs +graph).
3. **Which features matter most?** — Gain importance + optional SHAP for operations narrative.
4. **Which routes/hubs are hardest to predict?** — Error hotspot tables for targeted process fixes.

| Signal class | Examples |
|--------------|----------|
| OSRM / distance | `osrm_time`, `osrm_distance`, segment shares |
| Temporal | Hour (cyclic), weekend |
| Route mode | FTL vs Carting |
| **Graph** | `source_pagerank`, `destination_bottleneck_score`, community match |

## 1. Environment setup

In [1]:
from __future__ import annotations

import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from IPython.display import Markdown, display

from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from xgboost import XGBRegressor

warnings.filterwarnings("ignore", category=FutureWarning)

# =========================================================
# Global config
# =========================================================

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

sns.set_theme(
    style="whitegrid",
    context="notebook",
    palette="deep"
)

plt.rcParams.update({
    "figure.figsize": (11, 6),
    "figure.dpi": 110,
    "axes.titlesize": 12,
    "axes.labelsize": 10,
})

# =========================================================
# Project paths
# =========================================================

PROJECT_ROOT = Path.cwd()

# If running inside /notebooks
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

# VERY IMPORTANT
# Add ROOT project path first
sys.path.insert(0, str(PROJECT_ROOT))

# Add notebooks folder
sys.path.insert(0, str(PROJECT_ROOT / "notebooks"))

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

FIGURES_DIR = PROJECT_ROOT / "outputs" / "figures"

TABLES_DIR = PROJECT_ROOT / "outputs" / "tables"

for d in (FIGURES_DIR, TABLES_DIR):
    d.mkdir(parents=True, exist_ok=True)

NOTEBOOK_TAG = "04"

# =========================================================
# Notebook helper imports
# =========================================================

from _eta_modeling_lib import (
    build_lane_graph_tables,
    merge_hub_features,
    metrics_row,
    regression_metrics,
)

print("PROJECT ROOT:", PROJECT_ROOT)
print("Notebook imports loaded successfully.")

PROJECT ROOT: D:\delivery eta
Notebook imports loaded successfully.


### Helper utilities

In [2]:
def save_figure(fig: plt.Figure, name: str) -> Path:
    out = FIGURES_DIR / f"{NOTEBOOK_TAG}_{name}.png"
    fig.savefig(out, dpi=150, bbox_inches="tight")
    plt.close(fig)
    return out


def save_table(df: pd.DataFrame, name: str) -> Path:
    out = TABLES_DIR / f"{NOTEBOOK_TAG}_{name}.csv"
    df.to_csv(out, index=False)
    return out


def load_graph_tables_from_exports(tables_dir: Path) -> dict[str, pd.DataFrame]:
    """Load notebook 03 CSV exports into the dict expected by merge_hub_features."""
    pr = pd.read_csv(tables_dir / "03_hub_rankings_pagerank.csv")
    deg = pd.read_csv(tables_dir / "03_hub_rankings_degree.csv")
    bn = pd.read_csv(tables_dir / "03_bottleneck_hubs.csv")
    comm = pd.read_csv(tables_dir / "03_hub_communities.csv")
    return {
        "pagerank": pr[["hub_id", "pagerank"]].copy(),
        "degree": deg[["hub_id", "degree_centrality"]].copy(),
        "bottleneck": bn[["hub_id", "bottleneck_score"]].copy(),
        "communities": comm[["hub_id", "community_id", "community_size"]].copy(),
    }


def graph_hub_coverage(df: pd.DataFrame, tables: dict[str, pd.DataFrame]) -> float:
    hubs = pd.unique(df[["source_center", "destination_center"]].values.ravel("K"))
    covered = set(tables["pagerank"]["hub_id"].astype(str))
    return len([h for h in hubs if str(h) in covered]) / max(len(hubs), 1)

## 2. Load processed logistics data

Regenerate with: `python scripts/build_processed_dataset.py`

In [3]:
DATA_PATH = PROCESSED_DIR / "delivery_logistics_processed.parquet"
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Missing processed data: {DATA_PATH}")

df = pd.read_parquet(DATA_PATH)

required = {
    "trip_uuid",
    "source_center",
    "destination_center",
    "actual_time",
    "osrm_time",
    "osrm_distance",
    "segment_osrm_time",
    "segment_osrm_distance",
    "route_type",
    "route_lane",
    "trip_creation_time",
}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing columns: {sorted(missing)}")

df["trip_creation_time"] = pd.to_datetime(df["trip_creation_time"], utc=True, errors="coerce")
if "trip_hour" not in df.columns:
    df["trip_hour"] = df["trip_creation_time"].dt.hour
if "trip_dayofweek" not in df.columns:
    df["trip_dayofweek"] = df["trip_creation_time"].dt.dayofweek

print(f"Source: {DATA_PATH}")
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print("\nNull counts (key columns):")
display(df[list(required | {"trip_hour", "trip_dayofweek"})].isna().sum().to_frame("nulls"))
display(df[["actual_time", "osrm_time", "segment_osrm_time", "osrm_distance"]].describe().T)

Source: D:\delivery eta\data\processed\delivery_logistics_processed.parquet
Shape: 142,502 rows × 26 columns

Null counts (key columns):


,nulls
actual_time,0
osrm_time,0
segment_osrm_time,0
trip_creation_time,0
route_type,0
trip_dayofweek,0
route_lane,0
trip_hour,0
osrm_distance,0
source_center,0


,count,mean,std,min,25%,50%,75%,max
actual_time,142502.0,421.266319,600.444883,9.0000,52.00000,134.0000,525.000000,4532.0000
osrm_time,142502.0,216.238551,309.265760,6.0000,27.00000,66.0000,264.000000,1686.0000
segment_osrm_time,142502.0,18.811729,14.703428,1.0000,11.00000,17.0000,22.000000,1611.0000
osrm_distance,142502.0,288.006486,422.814551,9.0082,30.24075,79.8407,351.488875,2326.1991


## 3. Graph features — load or rebuild

Prefer **notebook 03 exports** (`outputs/tables/03_*.csv`). If hub coverage is below **80%**, rebuild from the full segment table via `build_lane_graph_tables`.

In [4]:
graph_files = [
    "03_bottleneck_hubs.csv",
    "03_hub_rankings_pagerank.csv",
    "03_hub_rankings_degree.csv",
    "03_hub_communities.csv",
]
if all((TABLES_DIR / f).exists() for f in graph_files):
    graph_tables = load_graph_tables_from_exports(TABLES_DIR)
    coverage = graph_hub_coverage(df, graph_tables)
    print(f"Loaded graph tables from {TABLES_DIR} — hub coverage: {coverage:.1%}")
else:
    coverage = 0.0
    graph_tables = None
    print("One or more 03_*.csv files missing; will rebuild graph tables.")

COVERAGE_THRESHOLD = 0.80
if graph_tables is None or coverage < COVERAGE_THRESHOLD:
    print("Rebuilding lane graph tables (build_lane_graph_tables)...")
    graph_tables = build_lane_graph_tables(df)
    coverage = graph_hub_coverage(df, graph_tables)
    print(f"Post-rebuild hub coverage: {coverage:.1%}")

df = merge_hub_features(df, graph_tables, hub_col="source_center", prefix="source")
df = merge_hub_features(df, graph_tables, hub_col="destination_center", prefix="destination")

graph_cols = [
    "source_pagerank", "destination_pagerank",
    "source_degree", "destination_degree",
    "source_bottleneck_score", "destination_bottleneck_score",
    "source_community", "destination_community",
]
display(df[graph_cols].describe().T)

Loaded graph tables from D:\delivery eta\outputs\tables — hub coverage: 3.0%
Rebuilding lane graph tables (build_lane_graph_tables)...
Post-rebuild hub coverage: 100.0%


,count,mean,std,min,25%,50%,75%,max
source_pagerank,142502.0,0.012531,0.015630,0.000113,0.000626,0.004259,0.018264,0.043584
destination_pagerank,142502.0,0.010750,0.013700,0.000115,0.000723,0.003661,0.015276,0.043584
source_degree,142502.0,0.021215,0.020536,0.000604,0.002415,0.012077,0.035024,0.056763
destination_degree,142502.0,0.019194,0.018670,0.000604,0.002415,0.011473,0.034420,0.056763
source_bottleneck_score,142502.0,0.249576,0.358440,0.000000,0.003469,0.052228,0.274891,1.000000
destination_bottleneck_score,142502.0,0.200756,0.308768,0.000000,0.003512,0.052228,0.244737,1.000000
source_community,142502.0,31.130412,34.454837,0.000000,10.000000,12.000000,45.000000,131.000000
destination_community,142502.0,33.136040,35.557513,0.000000,11.000000,12.000000,60.000000,131.000000


**Why graph features may improve ETA prediction**

OSRM encodes **geometry and speed limits**, not **operational friction** at chokepoint DCs.
**PageRank / degree** proxy hub importance in the flow network; **bottleneck score** flags structural congestion;
**community** captures regional sub-networks where delays covary.
Joining these at **source** and **destination** tells the model whether a trip touches stressed parts of the backbone—
improving predictions beyond distance and clock time alone.

## 4. Feature engineering

- **Temporal:** hour sin/cos, weekend flag
- **Route mode:** `route_type` dummies (FTL vs Carting)
- **Distance structure:** ratios from OSRM / segment fields (no `delay_ratio` / `segment_delay_ratio` — those leak the target)
- **Graph interactions:** products and contrasts of hub centrality / bottleneck
- **Robustness:** fill NaN/inf, clip numeric features at training 99th percentile

In [5]:
work = df.copy()

# Temporal encodings
work["trip_hour_sin"] = np.sin(2 * np.pi * work["trip_hour"] / 24)
work["trip_hour_cos"] = np.cos(2 * np.pi * work["trip_hour"] / 24)
work["is_weekend"] = (work["trip_dayofweek"] >= 5).astype(int)

# Route type dummies
rt = pd.get_dummies(work["route_type"].astype(str), prefix="route_type", dtype=float)
work = pd.concat([work, rt], axis=1)

# Distance / time ratios (OSRM-only — no post-hoc actuals)
work["segment_time_share"] = work["segment_osrm_time"] / work["osrm_time"].replace(0, np.nan)
work["segment_distance_share"] = work["segment_osrm_distance"] / work["osrm_distance"].replace(0, np.nan)
work["segment_osrm_to_trip_osrm_time"] = work["segment_osrm_time"] / work["osrm_time"].replace(0, np.nan)

# Graph interactions
work["graph_pagerank_product"] = work["source_pagerank"] * work["destination_pagerank"]
work["graph_bottleneck_sum"] = work["source_bottleneck_score"] + work["destination_bottleneck_score"]
work["graph_pagerank_gap"] = (work["source_pagerank"] - work["destination_pagerank"]).abs()
work["same_community"] = (work["source_community"] == work["destination_community"]).astype(float)

ROUTE_DUMMY_COLS = [c for c in work.columns if c.startswith("route_type_")]

BASE_FEATURES = [
    "osrm_time",
    "osrm_distance",
    "segment_osrm_time",
    "segment_osrm_distance",
    "trip_hour_sin",
    "trip_hour_cos",
    "is_weekend",
    "segment_time_share",
    "segment_distance_share",
    "segment_osrm_to_trip_osrm_time",
    *ROUTE_DUMMY_COLS,
]

GRAPH_FEATURES = [
    "source_pagerank",
    "destination_pagerank",
    "source_degree",
    "destination_degree",
    "source_bottleneck_score",
    "destination_bottleneck_score",
    "source_community",
    "destination_community",
    "graph_pagerank_product",
    "graph_bottleneck_sum",
    "graph_pagerank_gap",
    "same_community",
]

ALL_FEATURES = BASE_FEATURES + GRAPH_FEATURES
TARGET = "actual_time"

# Sanitize infinities / missing
numeric_feats = [c for c in ALL_FEATURES if c in work.columns]
work[numeric_feats] = work[numeric_feats].replace([np.inf, -np.inf], np.nan)
work[numeric_feats] = work[numeric_feats].fillna(work[numeric_feats].median())

print(f"BASE_FEATURES ({len(BASE_FEATURES)}): {BASE_FEATURES}")
print(f"GRAPH_FEATURES ({len(GRAPH_FEATURES)}): {GRAPH_FEATURES}")

BASE_FEATURES (12): ['osrm_time', 'osrm_distance', 'segment_osrm_time', 'segment_osrm_distance', 'trip_hour_sin', 'trip_hour_cos', 'is_weekend', 'segment_time_share', 'segment_distance_share', 'segment_osrm_to_trip_osrm_time', 'route_type_Carting', 'route_type_FTL']
GRAPH_FEATURES (12): ['source_pagerank', 'destination_pagerank', 'source_degree', 'destination_degree', 'source_bottleneck_score', 'destination_bottleneck_score', 'source_community', 'destination_community', 'graph_pagerank_product', 'graph_bottleneck_sum', 'graph_pagerank_gap', 'same_community']


## 5. Train / test split (by `trip_uuid`)

**Leakage warning:** A random row split would place segments from the same trip in both train and test, inflating scores because `actual_time` is trip-level and repeated across segments. We hold out **20% of trips** entirely.

In [6]:
trips = work["trip_uuid"].dropna().unique()
train_trips, test_trips = train_test_split(
    np.array(trips), test_size=0.2, random_state=RANDOM_STATE
)
train = work[work["trip_uuid"].isin(train_trips)].copy()
test = work[work["trip_uuid"].isin(test_trips)].copy()

print(
    f"Trips: train {len(train_trips):,} | test {len(test_trips):,} "
    f"(total {len(trips):,})"
)
print(f"Rows: train {len(train):,} | test {len(test):,}")

# Clip numeric features at train 99th percentile
clip_bounds = {}
for col in numeric_feats:
    hi = train[col].quantile(0.99)
    clip_bounds[col] = hi
    train[col] = train[col].clip(upper=hi)
    test[col] = test[col].clip(upper=hi)

X_train_base = train[BASE_FEATURES]
X_test_base = test[BASE_FEATURES]
X_train_all = train[ALL_FEATURES]
X_test_all = test[ALL_FEATURES]
y_train = train[TARGET]
y_test = test[TARGET]

Trips: train 11,853 | test 2,964 (total 14,817)
Rows: train 113,764 | test 28,738


## 6. Linear regression (scaled)

In [7]:
scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_train_all)
X_te_s = scaler.transform(X_test_all)

lin = LinearRegression()
lin.fit(X_tr_s, y_train)
y_pred_lin = lin.predict(X_te_s)

lin_metrics = metrics_row("LinearRegression", y_test, y_pred_lin)
display(pd.DataFrame([lin_metrics]))

,Model,MAE,RMSE,R2
0,LinearRegression,62.424524,135.354102,0.953801


**Interpretation:** Linear regression on scaled features establishes a **transparent baseline**.
Large gaps vs tree models indicate **non-linear interactions** (hub congestion × time-of-day) that justify XGBoost for production.

## 7. Random Forest & XGBoost — model comparison

In [8]:
rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=14,
    min_samples_leaf=5,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
rf.fit(X_train_all, y_train)
y_pred_rf = rf.predict(X_test_all)

xgb_full = XGBRegressor(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
xgb_full.fit(X_train_all, y_train)
y_pred_xgb = xgb_full.predict(X_test_all)

comparison = pd.DataFrame([
    lin_metrics,
    metrics_row("RandomForest", y_test, y_pred_rf),
    metrics_row("XGBoost+graph", y_test, y_pred_xgb),
])
display(comparison)
save_table(comparison, "model_comparison")

best_row = comparison.loc[comparison["MAE"].idxmin()]
print(f"Best model by test MAE: {best_row['Model']} (MAE={best_row['MAE']:.2f} min, R²={best_row['R2']:.4f})")

,Model,MAE,RMSE,R2
0,LinearRegression,62.424524,135.354102,0.953801
1,RandomForest,43.492476,112.589005,0.968035
2,XGBoost+graph,44.170184,111.940546,0.968402


Best model by test MAE: RandomForest (MAE=43.49 min, R²=0.9680)


**Interpretation:** Tree ensembles typically dominate when OSRM underfits **hub dwell** and **mode effects** (FTL vs Carting).
Persist `04_model_comparison.csv` for model governance and champion/challenger reviews.

## 8. XGBoost: without graph vs with graph

Isolates the marginal value of network features on the **same trip holdout**.

In [9]:
xgb_base = XGBRegressor(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
xgb_base.fit(X_train_base, y_train)
y_pred_base = xgb_base.predict(X_test_base)

xgb_graph = XGBRegressor(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
xgb_graph.fit(X_train_all, y_train)
y_pred_graph = xgb_graph.predict(X_test_all)

xgb_compare = pd.DataFrame([
    metrics_row("XGBoost (no graph)", y_test, y_pred_base),
    metrics_row("XGBoost (+ graph)", y_test, y_pred_graph),
])
display(xgb_compare)
save_table(xgb_compare, "xgb_graph_ablation")

m_base = xgb_compare.loc[xgb_compare["Model"] == "XGBoost (no graph)", "MAE"].iloc[0]
m_graph = xgb_compare.loc[xgb_compare["Model"] == "XGBoost (+ graph)", "MAE"].iloc[0]
delta = m_base - m_graph
print(
    f"Business insight: graph features change test MAE by {delta:+.2f} min "
    f"({'improvement' if delta > 0 else 'no improvement'} vs OSRM+tabular only)."
)

best_xgb = xgb_graph
y_pred_best = y_pred_graph

fig, ax = plt.subplots(figsize=(6, 4))
sns.barplot(
    data=xgb_compare, x="Model", y="MAE", ax=ax,
    hue="Model", palette="viridis", legend=False,
)
ax.set_title("Graph feature ablation — test MAE")
fig.tight_layout()
save_figure(fig, "xgb_graph_ablation_bar")

,Model,MAE,RMSE,R2
0,XGBoost (no graph),56.732419,130.210927,0.957245
1,XGBoost (+ graph),44.170184,111.940546,0.968402


Business insight: graph features change test MAE by +12.56 min (improvement vs OSRM+tabular only).


WindowsPath('D:/delivery eta/outputs/figures/04_xgb_graph_ablation_bar.png')

**Business insight (graph impact):** If **XGBoost (+ graph)** beats **XGBoost (no graph)** on MAE, network analytics from notebook 03
is not only descriptive—it is **predictive**. Invest in nightly graph refresh and graph-aware routing for high-bottleneck origins.
If lift is negligible, prioritize OSRM calibration and segment temporal features before embedding complexity.

## 9. Feature importance (best XGBoost)

In [10]:
importances = pd.Series(best_xgb.feature_importances_, index=ALL_FEATURES)
top_imp = importances.sort_values(ascending=False).head(20)

fig, ax = plt.subplots(figsize=(10, 8))
top_imp.sort_values().plot.barh(ax=ax, color="steelblue")
ax.set_title("Top 20 features — XGBoost (+ graph)")
ax.set_xlabel("Gain importance")
fig.tight_layout()
save_figure(fig, "feature_importance_top20")
display(top_imp.to_frame("importance"))

graph_in_top = [f for f in top_imp.index if f in GRAPH_FEATURES]
print(f"Graph features in top 20: {graph_in_top}")

,importance
osrm_distance,0.749569
osrm_time,0.187734
source_pagerank,0.006723
graph_bottleneck_sum,0.006405
graph_pagerank_product,0.005824
same_community,0.005687
graph_pagerank_gap,0.004104
source_degree,0.003935
source_bottleneck_score,0.003912
destination_bottleneck_score,0.003228


Graph features in top 20: ['source_pagerank', 'graph_bottleneck_sum', 'graph_pagerank_product', 'same_community', 'graph_pagerank_gap', 'source_degree', 'source_bottleneck_score', 'destination_bottleneck_score', 'destination_degree', 'destination_pagerank', 'source_community', 'destination_community']


**Interpretation:** OSRM columns usually rank highest (expected—router is the prior).
When **graph** features (e.g. `source_bottleneck_score`, `graph_bottleneck_sum`) appear in the top tier, ETA error is partly **structural**,
not just distance-driven—align with capacity investments from notebook 03.

## 10. Residual diagnostics

In [11]:
residuals = y_test - y_pred_best

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].hist(residuals, bins=60, color="steelblue", edgecolor="white")
axes[0].set_title("Residual distribution (test)")
axes[0].set_xlabel("actual − predicted (minutes)")

axes[1].scatter(y_test, y_pred_best, alpha=0.15, s=8, c="steelblue")
lim = [min(y_test.min(), y_pred_best.min()), max(y_test.max(), y_pred_best.max())]
axes[1].plot(lim, lim, "r--", lw=1)
axes[1].set_xlabel("Actual time")
axes[1].set_ylabel("Predicted time")
axes[1].set_title("Actual vs predicted")
fig.tight_layout()
save_figure(fig, "residual_diagnostics")

worst = test.assign(
    predicted=y_pred_best,
    residual=residuals,
    abs_error=np.abs(residuals),
).nlargest(15, "abs_error")[
    [
        "trip_uuid",
        "route_lane",
        "route_type",
        "source_center",
        "destination_center",
        "actual_time",
        "predicted",
        "abs_error",
    ]
]
display(worst)
save_table(worst, "worst_predictions")

,trip_uuid,route_lane,route_type,source_center,destination_center,actual_time,predicted,abs_error
118237,trip-153768779707932330,IND176303AAA → IND176310AAA,FTL,IND176303AAA,IND176310AAA,2491.0,80.394699,2410.605301
5494,trip-153761568647456368,IND271001AAA → IND271201AAA,Carting,IND271001AAA,IND271201AAA,2399.0,102.835564,2296.164436
78229,trip-153736469866480991,IND000000ACB → IND712311AAA,FTL,IND000000ACB,IND712311AAA,4152.0,2190.937988,1961.062012
78223,trip-153736469866480991,IND000000ACB → IND712311AAA,FTL,IND000000ACB,IND712311AAA,3734.0,1986.515991,1747.484009
78225,trip-153736469866480991,IND000000ACB → IND712311AAA,FTL,IND000000ACB,IND712311AAA,3802.0,2073.722168,1728.277832
78227,trip-153736469866480991,IND000000ACB → IND712311AAA,FTL,IND000000ACB,IND712311AAA,3880.0,2159.962402,1720.037598
78222,trip-153736469866480991,IND000000ACB → IND712311AAA,FTL,IND000000ACB,IND712311AAA,3694.0,1975.449585,1718.550415
78224,trip-153736469866480991,IND000000ACB → IND712311AAA,FTL,IND000000ACB,IND712311AAA,3770.0,2053.033447,1716.966553
78221,trip-153736469866480991,IND000000ACB → IND712311AAA,FTL,IND000000ACB,IND712311AAA,3640.0,1926.712891,1713.287109
78226,trip-153736469866480991,IND000000ACB → IND712311AAA,FTL,IND000000ACB,IND712311AAA,3854.0,2154.658203,1699.341797


WindowsPath('D:/delivery eta/outputs/tables/04_worst_predictions.csv')

**Interpretation:** Residual skew suggests **long-tail trips** (underprediction on very long `actual_time`).
Review worst rows for shared lanes or hubs—systematic bias warrants segment-specific calibration or hub SLA playbooks.

## 11. MAE by route type (FTL vs Carting)

In [12]:
by_route = []
for rt, grp in test.assign(pred=y_pred_best).groupby("route_type"):
    idx = grp.index
    by_route.append({
        "route_type": rt,
        "n": len(grp),
        "MAE": mean_absolute_error(y_test.loc[idx], grp["pred"]),
    })
mae_route = pd.DataFrame(by_route).sort_values("MAE", ascending=False)
display(mae_route)
save_table(mae_route, "mae_by_route_type")

fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(data=mae_route, x="route_type", y="MAE", ax=ax, palette="muted")
ax.set_title("Test MAE by route type")
fig.tight_layout()
save_figure(fig, "mae_by_route_type")

,route_type,n,MAE
1,FTL,20005,54.627822
0,Carting,8733,20.214491


WindowsPath('D:/delivery eta/outputs/figures/04_mae_by_route_type.png')

**Operational implication:** Higher MAE on one mode (often **Carting** with multi-segment consolidation) implies
**different model heads or calibration** per `route_type` in production, or separate cutoff policies by mode.

## 12. Error hotspots — hubs and lanes

In [13]:
eval_df = test.assign(pred=y_pred_best, abs_error=np.abs(y_test - y_pred_best))

hub_errors = (
    eval_df.groupby("source_center", as_index=False)
    .agg(segments=("abs_error", "count"), MAE=("abs_error", "mean"))
    .rename(columns={"source_center": "hub_id"})
    .nlargest(25, "MAE")
)
dest_errors = (
    eval_df.groupby("destination_center", as_index=False)
    .agg(segments=("abs_error", "count"), MAE=("abs_error", "mean"))
    .rename(columns={"destination_center": "hub_id"})
    .nlargest(25, "MAE")
)
route_errors = (
    eval_df.groupby("route_lane", as_index=False)
    .agg(segments=("abs_error", "count"), MAE=("abs_error", "mean"))
    .nlargest(25, "MAE")
)

display(Markdown("**Worst source hubs (by test MAE)**"))
display(hub_errors)
display(Markdown("**Worst destination hubs**"))
display(dest_errors)
display(Markdown("**Worst lanes**"))
display(route_errors)

save_table(hub_errors, "error_hotspots_source_hubs")
save_table(dest_errors, "error_hotspots_destination_hubs")
save_table(route_errors, "error_hotspots_routes")

**Worst source hubs (by test MAE)**

,hub_id,segments,MAE
125,IND176303AAA,3,813.008489
496,IND431131AAB,14,584.106115
237,IND271001AAA,5,470.781096
229,IND262405AAA,1,421.599159
504,IND431717AAA,1,412.022217
110,IND173212AAA,1,397.806335
363,IND370615AAB,1,318.076675
169,IND209801AAA,8,300.491651
1111,IND802212AAA,6,279.796524
620,IND507159AAA,2,257.756474


**Worst destination hubs**

,hub_id,segments,MAE
128,IND176310AAA,3,813.008489
181,IND221401AAA,1,477.326012
239,IND271201AAA,5,470.781096
262,IND282005AAA,1,205.804565
1056,IND781018AAB,218,171.068358
970,IND712311AAA,1268,161.844445
541,IND473226AAA,10,148.308173
190,IND232101AAB,3,147.083978
989,IND722101AAB,2,130.623440
506,IND431603AAA,81,128.874725


**Worst lanes**

,route_lane,segments,MAE
264,IND176303AAA → IND176310AAA,3,813.008489
828,IND431131AAB → IND431603AAA,14,584.106115
1007,IND507159AAA → IND507002AAA,1,490.440697
344,IND221313AAA → IND221401AAA,1,477.326012
414,IND271001AAA → IND271201AAA,5,470.781096
402,IND262405AAA → IND263153AAB,1,421.599159
841,IND431717AAA → IND431603AAA,1,412.022217
249,IND173212AAA → IND160002AAC,1,397.806335
92,IND110037AAM → IND781018AAB,68,342.115809
1262,IND580028AAA → IND421302AAG,1,334.267090


WindowsPath('D:/delivery eta/outputs/tables/04_error_hotspots_routes.csv')

**Hotspot narrative:** High-MAE **lanes** reflect unstable corridors (congestion unpredictability, poor OSRM fit).
High-MAE **hubs** align with bottleneck candidates—pair with live dwell telemetry and dynamic rerouting rules.
Filter tables with `segments >= 30` before executive escalation to avoid noise from rare lanes.

## 13. Production recommendations

| Capability | Recommendation |
|------------|----------------|
| **Real-time ETA** | Score XGB (+ graph) on trip creation; refresh when segment scan events arrive |
| **Graph-aware routing** | Penalize paths through top `bottleneck_score` hubs in route optimization |
| **Congestion monitoring** | Dashboard MAE + volume for `04_error_hotspots_*` tables |
| **Dynamic rerouting** | Trigger when predicted ETA exceeds SLA and alternate lane exists |
| **SLA risk** | Alert on trips touching high-MAE hubs in top decile of `source_bottleneck_score` |

**Engineering:** Version `ALL_FEATURES`, clip bounds, and graph tables with each model artifact in `models/`.

## 14. Executive summary

| Dimension | Outcome (fill after run) |
|-----------|-------------------------|
| **Best model** | Lowest test MAE in `04_model_comparison.csv` |
| **Best features** | Top of `04_feature_importance` / SHAP |
| **Graph impact** | ΔMAE from `04_xgb_graph_ablation.csv` |
| **Hard to predict** | `04_error_hotspots_routes.csv`, hub exports |

### Operational recommendations

1. Deploy **graph-augmented XGBoost** when ablation shows positive MAE lift; otherwise ship OSRM + temporal baseline first.
2. Run **trip-level** monitoring; never score with random segment splits in production backtests.
3. Target **top bottleneck hubs** from notebook 03 that also appear in error hotspots.
4. Separate **FTL vs Carting** calibration if MAE gap is material.

### Next engineering steps

- **Node2Vec** hub embeddings → replace hand-crafted graph scalars
- **Temporal graph learning** → weekly evolving centrality features
- **Real-time traffic APIs** → dynamic edge weights
- **Streaming inference** → Kafka/feature store with nightly graph batch join

### SHAP explainability

In [14]:
try:
    import shap

    explainer = shap.TreeExplainer(best_xgb)
    sample = X_test_all.sample(min(5000, len(X_test_all)), random_state=RANDOM_STATE)
    shap_values = explainer.shap_values(sample)

    fig, ax = plt.subplots(figsize=(10, 6))
    shap.summary_plot(shap_values, sample, show=False, max_display=20)
    fig = plt.gcf()
    save_figure(fig, "shap_summary")
    print("SHAP summary saved.")
except ImportError:
    print("SHAP not installed — skip with: pip install shap")

SHAP summary saved.


**SHAP interpretation:** Features pushing SHAP mass positive increase predicted ETA.
Use with operations to validate whether bottleneck and OSRM fields dominate on high-delay corridors.